# MT-GINO modular workflow
Model, data, training, transition, evaluation, and plotting code live under `FINAL/`. This notebook only assembles the modular API.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

ROOT = Path.cwd()
FINAL = ROOT / 'FINAL' if (ROOT / 'FINAL').is_dir() else ROOT / 'Flow-reconstruction-in-VPM-using-FNO' / 'FINAL'
sys.path.insert(0, str(FINAL))
from gino.utils import load_config
from gino.data.dataset import EvolutionDataset, load_processed_dataset, collate_one
from gino.data.normalization import NormalizationStats
from gino.data.reconstruction import rebuild_next_batch
from gino.model.gino import GINOSharedLatent, build_latent_grid
from gino.training.trainer import Trainer, set_deterministic_seed
from gino.dynamics.rollout import inference_rollout
from gino.evaluation.one_step import evaluate_one_step
from gino.evaluation.autoregressive import evaluate_autoregressive
from visualization.temporal import plot_temporal_errors

config_path = FINAL / 'configs/default.yaml'
cfg = load_config(config_path)
set_deterministic_seed(int(cfg['seed']))
device = torch.device('cuda' if cfg['device'] == 'auto' and torch.cuda.is_available() else ('cpu' if cfg['device'] == 'auto' else cfg['device']))
dataset_path = (FINAL / cfg['data']['dataset']).resolve()
data = load_processed_dataset(dataset_path)
features = cfg['data']['input_features']
global_features = cfg['data']['global_condition_channels']
feature_indices = [list(data['feature_names']).index(name) for name in features]
stats = NormalizationStats.from_dataset(data, feature_indices)


## Data splits and model
Normalization statistics come from the canonical preprocessing output, fitted on training rows only.


In [ ]:
max_particles, max_queries = cfg['data']['max_particles'], cfg['data']['max_queries']
horizon_max = max(cfg['training'].get('rollout_horizon_schedule', [1]))
def make_dataset(split):
    return EvolutionDataset(data, data[f'{split}_pair_ids'], features, global_features, max_particles, max_queries, horizon_max)
train_ds, val_ds, test_ds = make_dataset('train'), make_dataset('val'), make_dataset('test')
model_cfg = cfg['model']
model = GINOSharedLatent(len(features), len(data['target_names']), len(data['field_target_names']), len(global_features), model_cfg).to(device)
latent_grid = build_latent_grid(model_cfg['latent_res'], device)


## Train
One-step training remains the default. Enable rollout loss, pushforward, noise, or scheduled sampling in `configs/default.yaml` independently.


In [ ]:
run_dir = (FINAL / cfg['training']['run_dir']).resolve()
trainer = Trainer(model, latent_grid, train_ds, val_ds, stats, cfg['training'], run_dir, device)
norm_meta = {key: np.asarray(value).tolist() for key, value in vars(stats).items()}
metadata = {'dataset_path': str(dataset_path), 'normalization': norm_meta,
            'split': {key: np.asarray(data[key]).tolist() for key in ('train_pair_ids', 'val_pair_ids', 'test_pair_ids')},
            'feature_names': features, 'target_names': [str(x) for x in data['target_names']],
            'field_target_names': [str(x) for x in data['field_target_names']],
            'global_condition_channels': global_features}
history = trainer.fit(cfg, metadata)


## Evaluate and inspect
Rollout starts from a true state once, then rebuilds each subsequent input from predictions.


In [ ]:
from torch.utils.data import DataLoader
from gino.training.trainer import dataset_collate, move_batch
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_one)
one_step_records = evaluate_one_step(model, test_loader, latent_grid, stats, device)
initial = move_batch(collate_one([test_ds[0]]), device)
targets = initial['rollout_state_targets']
steps = min(max(cfg['evaluation']['rollout_horizons']), targets.shape[1])
eval_horizons = [h for h in cfg['evaluation']['rollout_horizons'] if h <= steps] or [1]
rebuild = lambda old, state, field: rebuild_next_batch(old, state, field, stats)
rollout_records = evaluate_autoregressive(model, initial, latent_grid, stats, targets[:, :steps],
    eval_horizons, rebuild)
print('One-step records:', len(one_step_records))
print('Rollout records:', rollout_records)
plot_temporal_errors(one_step_records, [dict(item, case='sample') for item in rollout_records], FINAL / 'notebook_results')
